# LineSight AI — первый YOLO11n baseline, ручной запуск

Свежая Google Colab VM: **Runtime → Change runtime type → T4 GPU**. Обучение запускается только в отдельной ячейке после START_TRAINING=True. Не используйте Run all перед проверкой данных и GPU.

Конфигурация: YOLO11n (COCO pretrained), imgsz=640, 25 эпох, batch 16 при ≥14 GiB свободной VRAM, иначе 8 (при ≥6 GiB), seed=20261103. Пороги диагностики фиксированы: conf=0.25, matching IoU=0.5. Test не используется моделью.

Цель — локальные повреждения керамики. Нет надёжных негативных изображений; возможны похожие сцены между split. Метрики внутреннего val не подтверждают качество на реальных видео.

Код приватного репозитория передаётся тем же форматом ZIP bundle с SHA256. Никаких GitHub-токенов, push или SSH. Этот notebook ещё не выполнялся в Colab UI.


In [ ]:
from pathlib import Path
import hashlib, json, sys, zipfile, subprocess
from google.colab import files
EXPECTED_BUNDLE_SHA256 = "44ad697cf694b0b8cfc959b2e1a3ab751b169b8f9e7b9033b7c7908a4eecd5b2"
uploaded = files.upload()  # выбрать только colab_baseline_v1_bundle.zip
assert len(uploaded) == 1
bundle_name, bundle_bytes = next(iter(uploaded.items()))
assert hashlib.sha256(bundle_bytes).hexdigest() == EXPECTED_BUNDLE_SHA256, 'Wrong or changed bundle'
work = Path('/content/linesight_baseline_v1')
work.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle_name) as archive:
    allowed = {'prepare_yolo.py','download_dataset.py','check_env.py','voc_dataset.py','plan.json','bundle_manifest.json','train.py','requirements-colab.txt','baseline_czjyz_ps_v1.json'}
    assert len(archive.infolist()) == len(allowed) and set(archive.namelist()) == allowed
    assert sum(i.file_size for i in archive.infolist()) < 30 * 1024**2
    metadata = json.loads(archive.read('bundle_manifest.json'))
    for name in allowed:
        data = archive.read(name)
        if name != 'bundle_manifest.json':
            assert hashlib.sha256(data).hexdigest() == metadata['files'][name]
        target = work / name
        assert not target.exists() or target.read_bytes() == data, "Existing code differs; use a fresh runtime"
        target.write_bytes(data)
del uploaded, bundle_bytes
sys.path.insert(0, str(work))
plan = json.loads((work/'plan.json').read_text())
print(plan['split_counts'], plan['split_objects'])


In [ ]:
# Установка только в Colab; локально эти GPU-зависимости не нужны.
subprocess.run([sys.executable,'-m','pip','install','-r',str(work/'requirements-colab.txt')],check=True)
import torch, torchvision, importlib.metadata
assert importlib.metadata.version('ultralytics') == '8.3.228'
assert torch.cuda.is_available(), 'Нет CUDA GPU — выберите T4 runtime; обучение на CPU запрещено'
props = torch.cuda.get_device_properties(0)
print('Actual GPU:',props.name,'VRAM GiB:',props.total_memory/1024**3)
print('torch',torch.__version__,'torchvision',torchvision.__version__,'CUDA build',torch.version.cuda)
# Один лёгкий CUDA/NMS smoke-test, без нейросети и весов.
torchvision.ops.nms(torch.tensor([[0.,0.,10.,10.]],device='cuda'),torch.tensor([.9],device='cuda'),.5)


Выберите источник исходного архива: upload (уже скачанный ZIP) либо hf (только один целевой файл). После скачивания обязательна строгая сверка SHA256 с планом. Если HF main изменился — остановиться; новый архив автоматически не принимается.


In [ ]:
SOURCE_MODE = 'upload'
if SOURCE_MODE == 'upload':
    uploaded_archive = files.upload()  # выбрать transmission line object detection.zip
    assert len(uploaded_archive) == 1
    archive_path = Path('/content') / next(iter(uploaded_archive))
    del uploaded_archive
elif SOURCE_MODE == 'hf':
    import importlib.util
    if importlib.util.find_spec('huggingface_hub') is None:
        subprocess.run([sys.executable,'-m','pip','install','huggingface_hub>=0.24,<2'],check=True)
    result = subprocess.run([sys.executable,str(work/'download_dataset.py'),'--download','--revision','main','--cache-dir','/content/hf-cache'],check=True,capture_output=True,text=True)
    archive_path = Path(json.loads(result.stdout)['archive'])
else:
    raise ValueError('Choose upload or hf')
from download_dataset import inspect_archive, safe_extract
archive_info = inspect_archive(archive_path)
assert archive_info['sha256'] == plan['source_archive']['sha256'], 'Archive differs from reviewed source; stop'
print('Archive verified:', archive_info['size_bytes'], 'bytes; unpacked:', archive_info['uncompressed_bytes'])
source = Path('/content/source_v1')
if not source.exists():
    source = safe_extract(archive_path, source)
else:
    print('Existing extraction reused; frozen dataset hashes will be checked next')


In [ ]:
dataset = Path('/content/yolo_czjyz_ps_v1')
if not dataset.exists():
    subprocess.run([sys.executable,str(work/'prepare_yolo.py'),'export','--plan',str(work/'plan.json'),'--source',str(source),'--output',str(dataset),'--hardlink'],check=True)
subprocess.run([sys.executable,str(work/'prepare_yolo.py'),'validate','--dataset',str(dataset)],check=True)
config = work/'baseline_czjyz_ps_v1.json'
subprocess.run([sys.executable,str(work/'train.py'),'preflight','--data',str(dataset/'data.yaml'),'--config',str(config)],check=True)


## Сохранение при отключении

Рекомендуется подключить Drive: сырые данные остаются в /content, на Drive копируются только результаты после первой эпохи, каждых пяти эпох и в конце. Полное отключение VM может потерять прогресс после последнего копирования. Без Drive автоматически сохраняется локальный ZIP, который нужно скачать до сброса VM.


In [ ]:
USE_DRIVE_BACKUP = True  # False: только локальный ZIP, без защиты от полного сброса VM
RUN_ID = 'baseline640_v1'  # каждый повтор — новое имя, без перезаписи предыдущих результатов
BATCH = None  # автоматически 16/8 по свободной VRAM; при CUDA OOM: новое RUN_ID и BATCH=8
run_dir = Path('/content/linesight_runs') / RUN_ID
backup_dir = None
if USE_DRIVE_BACKUP:
    from google.colab import drive
    drive.mount('/content/drive')  # подключение аккаунта выполняет пользователь
    backup_dir = Path('/content/drive/MyDrive/LineSightAI/runs') / RUN_ID
assert not run_dir.exists(), 'Локальный run уже существует: задайте новый RUN_ID'
assert backup_dir is None or not backup_dir.exists(), 'Run на Drive уже существует: задайте новый RUN_ID'


## Ручной старт

Проверьте GPU, число изображений 3109/389/389 и место сохранения. Только затем измените START_TRAINING на True и выполните следующую ячейку. Если установка библиотек потребует Restart session, повторите ячейки сверху: существующие скрипты сверяются, распаковка и экспорт повторно не выполняются, набор снова проверяется. Каждый повтор обучения требует нового RUN_ID.


In [ ]:
START_TRAINING = False  # изменить вручную после проверок
if START_TRAINING:
    command = [sys.executable,str(work/'train.py'),'run','--data',str(dataset/'data.yaml'),'--config',str(config),'--output',str(run_dir)]
    if backup_dir is not None:
        command += ['--backup-dir',str(backup_dir)]
    if BATCH is not None:
        command += ['--batch',str(BATCH)]
    subprocess.run(command,check=True)
else:
    print('Обучение не запущено. Установите START_TRAINING=True для ручного старта.')


In [ ]:
# После обучения или прерывания с сохранёнными локальными файлами.
import shutil
zip_path = run_dir.with_suffix('.zip')
if run_dir.exists():
    subprocess.run([sys.executable,str(work/'train.py'),'export','--output',str(run_dir)],check=True)
    if backup_dir is not None:
        backup_dir.mkdir(parents=True,exist_ok=True)
        shutil.copy2(zip_path,backup_dir/zip_path.name)
    files.download(str(zip_path))
else:
    print('Run ещё не создан; обучение не выполнялось.')


## Вернуть результаты Codex

Скачанный baseline640_v1.zip положите в ignored локальный ml/runs/. В ZIP: training/weights/best.pt, last.pt и периодические epoch*.pt, results.csv, args.yaml, plots/confusion matrix, validation_metrics.json, val_predictions.json, examples/, error_analysis.json, manifest.json, dataset_manifest.json, effective_config.json, environment.json, pip-freeze.txt и training.log. Датасет в ZIP не включается.

TP/FP/FN — сравнение с аннотациями val, а не экспертная оценка. Если примеров категории нет, они не выдумываются. Test остаётся для отдельной оценки после согласования конфигурации.

При ошибке проверьте errors.txt и backup_errors.txt. При CUDA OOM сохраните неудачный run, выберите BATCH=8 и новое RUN_ID; не меняйте split или test. При сбросе VM сохранённые на Drive файлы остаются. Периодические epoch*.pt предназначены для восстановления; финальные best/last могут быть без optimizer state. Не обещается точное возобновление без проверки конкретного чекпоинта и окружения.
